In [4]:
# ============================================================
# ⚙️ NOTEBOOK 2 — Data Preprocessing (Clean Version)
# ============================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# ── Setup Paths ──────────────────────────────────────────────
BASE_DIR    = os.path.expanduser('~/customer-churn-prediction')
DATA_DIR    = os.path.join(BASE_DIR, 'data')
MODELS_DIR  = os.path.join(BASE_DIR, 'models')

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("✅ BASE_DIR  :", BASE_DIR)
print("✅ DATA_DIR  :", DATA_DIR)
print("✅ MODELS_DIR:", MODELS_DIR)

# ── Load Data ────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'WA_Fn-UseC_-Telco-Customer-Churn.csv'))
print(f"\n📌 Loaded data: {df.shape}")

# ── Fix TotalCharges ─────────────────────────────────────────
# ✅ Fix 1: Use direct assignment instead of inplace=True
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# ── Drop customerID ──────────────────────────────────────────
df = df.drop('customerID', axis=1)

# ── Encode Target ────────────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Encode Categorical Columns ───────────────────────────────
# ✅ Fix 2: Use include='str' instead of include='object'
cat_cols = df.select_dtypes(include='str').columns.tolist()
print("📌 Categorical columns:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# ── Split Features & Target ──────────────────────────────────
X = df.drop('Churn', axis=1)
y = df['Churn']

# ── Scale Numerical Features ─────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train/Test Split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📌 Train size       : {X_train.shape}")
print(f"📌 Test size        : {X_test.shape}")
print(f"📌 Churn rate train : {y_train.mean():.2%}")
print(f"📌 Churn rate test  : {y_test.mean():.2%}")

# ── Save Scaler ──────────────────────────────────────────────
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print("\n✅ Scaler saved!")

# ── Save Processed Data ──────────────────────────────────────
pd.DataFrame(X_train).to_csv(os.path.join(DATA_DIR, 'X_train.csv'), index=False)
pd.DataFrame(X_test).to_csv(os.path.join(DATA_DIR,  'X_test.csv'),  index=False)
y_train.to_csv(os.path.join(DATA_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(DATA_DIR,  'y_test.csv'),  index=False)

print("✅ Processed data saved!")
print("\n✅ Preprocessing Complete!")
print(f"📁 Files saved to: {DATA_DIR}")

✅ BASE_DIR  : /Users/pravallikachepuri/customer-churn-prediction
✅ DATA_DIR  : /Users/pravallikachepuri/customer-churn-prediction/data
✅ MODELS_DIR: /Users/pravallikachepuri/customer-churn-prediction/models

📌 Loaded data: (7043, 21)
📌 Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

📌 Train size       : (5634, 19)
📌 Test size        : (1409, 19)
📌 Churn rate train : 26.54%
📌 Churn rate test  : 26.54%

✅ Scaler saved!
✅ Processed data saved!

✅ Preprocessing Complete!
📁 Files saved to: /Users/pravallikachepuri/customer-churn-prediction/data
